# An Interpreter for a Simple Programming Language: Exercise Frame

This notebook is the starting point for the exercise at the end of the chapter *Implementing a Simple
Interpreter*.  It contains the interpreter developed in the notebook `03-Interpreter.ipynb`, extended by
*floating point numbers*, *strings*, and a number of *predefined functions*.  Your task is to extend this
interpreter as follows:

1. Add `for` loops to the interpreter.
2. Add the logical operators `&&` (*and*), `||` (*or*), and `!` (*not*).  The operator `!` should bind
   strongest and the operator `||` should bind weakest.
3. Add user-defined functions:
   - A function should only have access to its parameters and to those variables that are defined
     locally inside the function.
   - A function should always return a value using a `return` statement.  As a `return` statement can
     occur anywhere in a function, use an *exception* to communicate the return value and to transfer
     the control flow out of the function.

The places where you have to add code are marked with the comment `your code here`.  The notebook runs
as it is; only the example programs at the end need your extensions.

In [ ]:
from lark import Lark, Token
import math

The external helper file `AST2Dot.ipynb` is used to represent the nested tuples that serve as our
abstract syntax trees as graphs.

In [ ]:
%run AST2Dot.ipynb

## Specification of the Grammar

Compared to the grammar of `03-Interpreter.ipynb`, the grammar below has the following additions:
- Assignments are described by the syntactical variable `assign`, so that they can also be used inside
  the head of a `for` loop.
- The tokens `FLOAT` and `STRING` describe floating point numbers and string literals.

Add the grammar rules for `for` loops, the logical operators, function definitions, and `return`
statements.  Remember to give your rules *aliases*: Then `ast_to_tuples` only needs new entries in the
dictionary `SYMBOL_MAP`.  For the logical operators, introduce the syntactical variables
`disjunction`, `conjunction`, and `negation` in order to encode the precedences.

In [ ]:
sl_grammar = r"""
?start: program

program: stmnt* -> prog

?stmnt: "if" "(" bool_expr ")" stmnt       -> if_stmnt
      | "while" "(" bool_expr ")" stmnt    -> while_stmnt
      // your code here: for loops
      // your code here: function definitions and return statements
      | "{" program "}"                    -> block
      | assign ";"
      | expr ";"                           -> expr_stmnt

assign: IDENTIFIER ":=" expr

// your code here: the logical operators ||, &&, and !
?bool_expr: expr "==" expr -> eq
          | expr "!=" expr -> ne
          | expr "<=" expr -> le
          | expr ">=" expr -> ge
          | expr "<" expr  -> lt
          | expr ">" expr  -> gt

?expr: expr "+" product -> add
     | expr "-" product -> sub
     | product

?product: product "*" factor -> mul
        | product "/" factor -> div
        | product "%" factor -> mod
        | factor

?factor: "(" expr ")"
       | NUMBER                       -> number
       | FLOAT                        -> float_number
       | STRING                       -> string
       | IDENTIFIER                   -> var
       | IDENTIFIER "(" expr_list ")" -> fct_call

expr_list: (expr ("," expr)*)? -> explist

// Lexical definitions
%import common.CNAME -> IDENTIFIER
%import common.WS

FLOAT:  /[0-9](_?[0-9])*\.[0-9](_?[0-9])*([eE][+-]?[0-9]+)?/
NUMBER: /[0-9](_?[0-9])*/
STRING: /"(\\.|[^"\\])*"/

// In order to show how it's done, I have defined the tokens
// CPP_COMMENT and C_COMMENT describing comments myself.
CPP_COMMENT: /\/\/[^\n]*/
C_COMMENT:   "/*" /(.|\n)*?/ "*/"

// Ignore whitespace and comments
%ignore WS
%ignore C_COMMENT
%ignore CPP_COMMENT
"""

We instantiate the parser utilizing the *Look-Ahead Left-to-Right* (LALR) algorithm.

In [ ]:
parser = Lark(sl_grammar, parser='lalr')

## Generating the Abstract Syntax Tree

Our abstract syntax trees are nested tuples.  Numbers are represented as `int` or `float`, variables as
strings.  A string literal is represented by the string *including* its double quotes, so that it can be
distinguished from a variable.

In [ ]:
Number      = int | float
Value       = Number | str
NestedTuple = AST

`SYMBOL_MAP` is a dictionary mapping grammar aliases to their symbols in the abstract syntax tree.

In [ ]:
SYMBOL_MAP = {
    # Program and Statements
    'prog': '.',
    'if_stmnt': 'if',
    'while_stmnt': 'while',
    'expr_stmnt': 'expr',
    # your code here: for loops, function definitions, and return statements
    # Comparison Operators
    'eq': '==', 'ne': '!=', 'le': '<=', 'ge': '>=', 'lt': '<', 'gt': '>',
    # your code here: the logical operators
    # Arithmetic Operations
    'add': '+', 'sub': '-', 'mul': '*', 'div': '/', 'mod': '%'
}

`ast_to_tuples` takes a node of the syntax tree and transforms it into a nested tuple.

In [ ]:
def ast_to_tuples(node) -> NestedTuple:
    # Base case: Leaf node (Token)
    if isinstance(node, Token):
        if node.type == 'NUMBER':
            return int(node.value)
        if node.type == 'FLOAT':
            return float(node.value)
        return str(node.value)
    if node.data in SYMBOL_MAP:
        symbol = SYMBOL_MAP[node.data]
        return (symbol, *[ast_to_tuples(child) for child in node.children])
    match node.data:
        case 'block':
            return ast_to_tuples(node.children[0])
        case 'assign':
            return (':=', str(node.children[0]), ast_to_tuples(node.children[1]))
        case 'number':
            return int(node.children[0])
        case 'float_number':
            return float(node.children[0])
        case 'string':
            return str(node.children[0])
        case 'var':
            return str(node.children[0])
        case 'fct_call':
            f_name = str(node.children[0])
            args   = ast_to_tuples(node.children[1])
            return ('call', f_name, *args)
        case 'explist':
            return tuple([ast_to_tuples(child) for child in node.children])
        # your code here: the list of parameters of a function definition
        case _:
            raise ValueError(f"Unknown syntax tree node: {node.data}")

The function `parse` takes a `file_name` as its sole argument.  The file is read, parsed, and
transformed into an abstract syntax tree, which is then displayed as a graph.

In [ ]:
def parse(file_name: str):
    with open(file_name, 'r') as handle:
        program = handle.read()
    print(program)
    tree = parser.parse(program)
    ast  = ast_to_tuples(tree)
    print(ast)
    return tuple2dot(ast)

In [ ]:
parse('sum.sl')

## The Runtime System

A `return` statement can occur anywhere inside a function.  Therefore, it is implemented by raising an
exception of the class `ReturnValue`, which carries the value that is returned.  The call of the function
catches this exception.

In [ ]:
class ReturnValue(Exception):
    def __init__(self, value: Value):
        self.value = value

The dictionary `Functions` stores the user-defined functions.  It maps the name of a function to a pair
consisting of the list of its parameters and its body.

In [ ]:
Functions: dict[str, tuple[list[str], NestedTuple]] = {}

The function `execute_tuple` executes a list of statements.  The dictionary `Values` maps the names of
variables to their values.

In [ ]:
def execute_tuple(StatementList, Values: dict[str, Value]) -> None:
    for stmnt in StatementList:
        execute(stmnt, Values)

The function `execute` matches recursively against the shape of the nested tuple that represents a
statement.

In [ ]:
def execute(stmnt: NestedTuple, Values: dict[str, Value]) -> None:
    match stmnt:
        case ('.', *SL):
            execute_tuple(SL, Values)
        case (':=', var, value):
            Values[var] = evaluate(value, Values)
        case ('expr', expr):
            evaluate(expr, Values)
        case ('if', test, stmnt):
            if evaluate_bool(test, Values):
                execute(stmnt, Values)
        case ('while', test, stmnt):
            while evaluate_bool(test, Values):
                execute(stmnt, Values)
        # your code here: for loops
        # your code here: function definitions
        # your code here: return statements (raise a ReturnValue exception)
        case _:
            assert False, f'{stmnt} unexpected'

The function `evaluate_bool` evaluates the test of a conditional or of a loop and returns a Boolean
value.

In [ ]:
def evaluate_bool(expr: NestedTuple, Values: dict[str, Value]) -> bool:
    match expr:
        case ('==', lhs, rhs):
            return evaluate(lhs, Values) == evaluate(rhs, Values)
        case ('!=', lhs, rhs):
            return evaluate(lhs, Values) != evaluate(rhs, Values)
        case ('<=', lhs, rhs):
            return evaluate(lhs, Values) <= evaluate(rhs, Values)
        case ('>=', lhs, rhs):
            return evaluate(lhs, Values) >= evaluate(rhs, Values)
        case ('<', lhs, rhs):
            return evaluate(lhs, Values) <  evaluate(rhs, Values)
        case ('>', lhs, rhs):
            return evaluate(lhs, Values) >  evaluate(rhs, Values)
        # your code here: the logical operators &&, ||, and !
        case _:
            assert False, f'{expr} unexpected'

The function `evaluate` computes the value of an expression.  A call of a function is either a call of a
user-defined function or a call of a predefined function.

In [ ]:
def evaluate(expr: NestedTuple, Values: dict[str, Value]) -> Value:
    match expr:
        case int() | float():
            return expr
        case str() if expr.startswith('"'):
            return expr[1:-1]
        case str():
            return Values[expr]
        case ('+', lhs, rhs):
            return evaluate(lhs, Values) + evaluate(rhs, Values)
        case ('-', lhs, rhs):
            return evaluate(lhs, Values) - evaluate(rhs, Values)
        case ('*', lhs, rhs):
            return evaluate(lhs, Values) * evaluate(rhs, Values)
        case ('/', lhs, rhs):
            return evaluate(lhs, Values) / evaluate(rhs, Values)
        case ('%', lhs, rhs):
            return evaluate(lhs, Values) % evaluate(rhs, Values)
        case ('call', f_name, *Args) if f_name in Functions:
            # your code here: call a user-defined function
            #  - evaluate the arguments,
            #  - execute the body with a new dictionary that contains only the parameters,
            #  - the body returns its value by raising a ReturnValue exception.
            pass
        case ('call', f_name, *Args):
            return call_builtin(f_name, [evaluate(arg, Values) for arg in Args])
        case _:
            assert False, f'{expr} unexpected'

The function `call_builtin` implements the predefined functions of our language.  The function `print`
prints all of its arguments without separators between them.

In [ ]:
def call_builtin(f_name: str, Args: list[Value]) -> Value:
    match f_name:
        case 'read':
            s = input('Please enter a number: ')
            return float(s) if '.' in s or 'e' in s or 'E' in s else int(s)
        case 'print':
            print(*Args, sep='')
            return 0
        case 'sqrt':
            return math.sqrt(Args[0])
        case 'exp':
            return math.exp(Args[0])
        case 'ln':
            return math.log(Args[0])
        case 'sin':
            return math.sin(Args[0])
        case 'cos':
            return math.cos(Args[0])
        case 'tan':
            return math.tan(Args[0])
        case 'arctan':
            return math.atan(Args[0])
        case _:
            assert False, f'function {f_name} is unknown'

## Putting Everything Together

The function `main` reads a program from a file, parses it, and executes it.  If `show_ast` is `True`, the
abstract syntax tree is displayed first.

In [ ]:
def main(file_name: str, show_ast: bool = False) -> None:
    with open(file_name, 'r') as handle:
        program = handle.read()
    tree = parser.parse(program)
    ast  = ast_to_tuples(tree)
    if show_ast:
        display(tuple2dot(ast))
    Functions.clear()
    execute(ast, {})

The program `sum.sl` only uses features that are already implemented.

In [ ]:
!cat sum.sl

In [ ]:
main('sum.sl')

## Testing Your Extensions

The following programs need your extensions.  The program `factorial.sl` uses user-defined functions and
a `for` loop.

In [ ]:
!cat Exercise/factorial.sl

In [ ]:
main('Exercise/factorial.sl')

In [ ]:
!cat Exercise/e.sl

In [ ]:
main('Exercise/e.sl')

In [ ]:
!cat Exercise/pi.sl

In [ ]:
main('Exercise/pi.sl')

In [ ]:
!cat Exercise/solve.sl

In [ ]:
main('Exercise/solve.sl')

In [ ]:
!cat Exercise/sum-for.sl

In [ ]:
main('Exercise/sum-for.sl')